# 00 - ViT dynamic-LRP repro (toolchain-proof gate)

Reproduces the reference `keeinlev/dynamicLRP` `src/experiments/ViT.ipynb`
attribution flow on the **pinned** MapClass stack (torch==2.7.1,
transformers==4.52.3, dynamicLRP vendored at SHA
`405e74243ecaa1f615f418fdc8ba24c3c5889b1e`).

This is ROADMAP Success Criterion 1 / Plan 01-01 Task 2. The SigLIP-2 swap
(Plan 03) must NOT begin until this notebook reproduces a structured ViT
relevance heatmap on this exact environment.

Flow mirrors the reference notebook cells 1-15 verbatim in mechanism
(`ViTForImageClassification` 224/patch16, one CIFAR10 image, `LRPEngine(
use_gamma=True, no_recompile=True)`, `params_to_interpret=[img_tensor]`,
`run(output.logits)`).

In [ ]:
# --- Wire the vendored dynamicLRP engine onto sys.path (D-08: in-tree, offline) ---
import os
import sys
from pathlib import Path

# notebooks/ -> repo root
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
VENDORED_LRP_SRC = REPO_ROOT / "third_party" / "dynamicLRP" / "src"
PROJECT_SRC = REPO_ROOT / "src"
for p in (str(VENDORED_LRP_SRC), str(PROJECT_SRC)):
    if p not in sys.path:
        sys.path.insert(0, p)

vendor_sha = (REPO_ROOT / "third_party" / "dynamicLRP" / "VENDOR_SHA").read_text().strip()
print("repo root        :", REPO_ROOT)
print("vendored LRP src :", VENDORED_LRP_SRC)
print("VENDOR_SHA       :", vendor_sha)
assert vendor_sha == "405e74243ecaa1f615f418fdc8ba24c3c5889b1e", vendor_sha

In [ ]:
import random

import numpy as np
import torch
import torchvision.datasets as datasets
import torchvision.transforms as T
import transformers
from matplotlib import pyplot as plt
from transformers import ViTConfig, ViTForImageClassification

# Pinned-stack assertion (fail loudly if the env drifted)
assert torch.__version__.startswith("2.7.1"), torch.__version__
assert transformers.__version__ == "4.52.3", transformers.__version__
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda        :", torch.cuda.is_available())

seed_value = 42
torch.manual_seed(seed_value)
np.random.seed(seed_value)
random.seed(seed_value)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Vendored engine import (offline; no network, no clone-at-setup)
from lrp_engine import LRPEngine

print("LRPEngine imported from:", LRPEngine.__module__)
assert hasattr(LRPEngine, "run") and hasattr(LRPEngine, "get_model_operations")

In [ ]:
# Reference model: HF ViTForImageClassification, patch16 / 224 (ViT.ipynb cell 3)
vit_model = ViTForImageClassification.from_pretrained(
    "nateraw/vit-base-patch16-224-cifar10"
)
vit_model.to(device)
vit_model.eval()
print("model loaded:", type(vit_model).__name__)

In [ ]:
# One sample image, reference transform (ViT.ipynb cell 6)
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
dataset = datasets.CIFAR10(
    root=str(REPO_ROOT / ".cache" / "cifar10"),
    train=False,
    download=True,
    transform=transform,
)
sample_img, sample_label = dataset[0]
print("sample image tensor:", tuple(sample_img.shape), "label:", sample_label)

In [ ]:
# Forward + coverage probe (ViT.ipynb cells 11-12)
# img_tensor is the SAME object passed to both the forward and params_to_interpret.
img_tensor = sample_img.unsqueeze(0).to(device).requires_grad_()
output = vit_model(img_tensor)

op_names, op_count, _graph = LRPEngine.get_model_operations(output.logits)
print("get_model_operations op count:", op_count)
assert op_count > 0, "coverage probe returned no autograd ops"

In [ ]:
# Dynamic LRP (ViT.ipynb cells 10, 13) - target is the 2-D classification logits
engine = LRPEngine(use_gamma=True, no_recompile=True)
engine.params_to_interpret = [img_tensor]
ckpt_vals, param_vals = engine.run(output.logits)

relevance = param_vals[0]            # positionally matches params_to_interpret[0]
print("relevance type :", type(relevance))
print("relevance shape:", tuple(relevance.shape))
assert tuple(relevance.shape) == tuple(img_tensor.shape), (
    relevance.shape, img_tensor.shape
)

In [ ]:
# Per-patch grid reshape - verified ViT geometry: 224 / 16 = 14 -> 14x14 patches
patch_size = 16
img_dims = 224
grid = img_dims // patch_size            # = 14
assert img_dims % patch_size == 0

# (1,3,224,224) -> abs over channels -> (224,224) pixel relevance
pixel_rel = relevance.detach().abs().sum(1)[0]
# block-reduce 224x224 -> 14x14 (per-patch mean)
patch_grid = (
    pixel_rel.reshape(grid, patch_size, grid, patch_size).mean((1, 3))
)
print("patch grid shape:", tuple(patch_grid.shape))
assert tuple(patch_grid.shape) == (14, 14), patch_grid.shape

In [ ]:
# Visualize: reference raw heatmap (.sum over channels, ViT.ipynb cell 15)
# alpha-blended over the sample image, plus the 14x14 patch grid.
raw_heatmap = relevance.detach()[0].sum(0).cpu().numpy()

# de-normalize the sample image just for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
disp_img = (sample_img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(disp_img)
axs[0].set_title(f"CIFAR10 sample (label {sample_label})")
axs[0].set_axis_off()

axs[1].imshow(disp_img)
axs[1].imshow(raw_heatmap, cmap="bwr", alpha=0.5)
axs[1].set_title("Dynamic-LRP relevance (overlay)")
axs[1].set_axis_off()

im = axs[2].imshow(patch_grid.cpu().numpy(), cmap="bwr", interpolation="nearest")
axs[2].set_title("14x14 per-patch relevance")
axs[2].set_axis_off()
fig.colorbar(im, ax=axs[2], fraction=0.046)
fig.suptitle(
    f"ViT dynamic-LRP repro | torch {torch.__version__} | "
    f"transformers {transformers.__version__} | LRP @ {vendor_sha[:12]}"
)
fig.tight_layout()
plt.show()

# Structure probe (NOT a gate - the human eyeball is the gate): a meaningful
# attribution is non-uniform. Printed for context only.
print("relevance std :", float(relevance.detach().std()))
print("patch grid min/max:", float(patch_grid.min()), float(patch_grid.max()))

## Human-verify gate (Plan 01-01 Task 3)

Restart Kernel & Run All against the `mapclass (.venv)` kernel and confirm the
overlay/patch-grid heatmap is **structured and non-uniform** (concentrates on
the object, not flat / not pure noise) - matching the qualitative look of the
reference dynamicLRP `ViT.ipynb` output. If so, the pinned toolchain is proven
and Plan 03 (SigLIP-2 swap) may proceed.